# Exploratory Data Analysis — OpenWebText

**Tiny LLM Challenge · Milestone 1: Data Preprocessing + EDA**

This notebook streams a representative sample from `Skylion007/openwebtext` (~8M documents, ~13.5 GB compressed) via HuggingFace `datasets` in **streaming mode** — zero disk usage.

---
## 0 · Setup & Configuration

In [ ]:
import re
import time
import sys
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import GPT2TokenizerFast

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.9)

# set seed
def set_seed(seed):
    np.random.seed(seed)
    import random

    random.seed(seed)

set_seed(42)

In [ ]:
# ── Sample sizes ──────────────────────────────────────────────
N_LARGE = 20_000   # string-level statistics
N_SMALL = 5_000    # tokenization (heavier)

# ── Model constraints (Tiny LLM challenge) ────────────────────
CONTEXT_LENGTH = 1024
VOCAB_SIZE = 50_257

# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print(f"Tokenizer ready — vocab size: {tokenizer.vocab_size:,}")

---
## 1 · Dataset Overview

In [ ]:
ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

print("Dataset object:", ds)
print("Features:      ", ds.features)

In [ ]:
print("=" * 70)
print("SAMPLE DOCUMENTS (first 5)")
print("=" * 70)

for i, example in enumerate(ds.take(5)):
    text = example["text"]
    preview = text[:500].replace("\n", "\\n")
    print(f"\n── Doc {i} ({len(text):,} chars) ──")
    print(preview)
    if len(text) > 500:
        print("...")

In [ ]:
print(f"Streaming {N_LARGE:,} documents into memory ...")
t0 = time.time()
texts = [ex["text"] for ex in ds.take(N_LARGE)]
elapsed = time.time() - t0
mem_mb = sum(sys.getsizeof(t) for t in texts) / 1e6
print(f"Done in {elapsed:.1f}s — {len(texts):,} docs, ~{mem_mb:.1f} MB in memory")

---
## 2 · Document Length Distributions

In [ ]:
char_counts  = np.array([len(t) for t in texts])
word_counts  = np.array([len(t.split()) for t in texts])
sent_counts  = np.array([len(re.split(r'[.!?]+', t)) for t in texts])

# Summary table
percentiles = [1, 5, 25, 50, 75, 95, 99]
rows = []
for name, arr in [("Characters", char_counts), ("Words", word_counts), ("Sentences", sent_counts)]:
    row = {"Metric": name, "Mean": f"{arr.mean():,.0f}", "Std": f"{arr.std():,.0f}"}
    for p in percentiles:
        row[f"P{p}"] = f"{np.percentile(arr, p):,.0f}"
    rows.append(row)

df_lengths = pd.DataFrame(rows).set_index("Metric")
df_lengths

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, arr, label, color in zip(
    axes,
    [char_counts, word_counts, sent_counts],
    ["Characters", "Words", "Sentences"],
    ["steelblue", "seagreen", "coral"],
):
    clip = np.percentile(arr, 99)
    sns.histplot(arr[arr <= clip], bins=80, color=color, edgecolor="white",
                 linewidth=0.3, ax=ax)
    ax.set_title(f"{label} per doc (clipped at P99)")
    ax.set_xlabel(label)
    ax.axvline(np.median(arr), color="black", ls="--", lw=1,
               label=f"median={np.median(arr):,.0f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 3 · Tokenization Analysis (GPT-2 BPE)

In [ ]:
small_texts = texts[:N_SMALL]

print(f"Tokenizing {N_SMALL:,} documents ...")
t0 = time.time()
token_ids_list = tokenizer(small_texts, add_special_tokens=False)["input_ids"]
tok_elapsed = time.time() - t0
print(f"Done in {tok_elapsed:.1f}s")

token_counts = np.array([len(ids) for ids in token_ids_list])
small_word_counts = word_counts[:N_SMALL]
tpw_ratios = token_counts / np.maximum(small_word_counts, 1)  # tokens per word

print(f"\nToken counts — mean: {token_counts.mean():,.0f}, median: {np.median(token_counts):,.0f}")
print(f"Tokens/word  — mean: {tpw_ratios.mean():.2f}, median: {np.median(tpw_ratios):.2f}")

# Vocab coverage sanity check
unique_ids_seen = len({tid for ids in token_ids_list for tid in ids})
print(f"Vocab coverage: {unique_ids_seen:,}/{VOCAB_SIZE:,} IDs seen ({unique_ids_seen/VOCAB_SIZE*100:.1f}%)")

In [ ]:
fits_context = (token_counts <= CONTEXT_LENGTH).sum()
needs_chunking = (token_counts > CONTEXT_LENGTH).sum()
pct_fits = fits_context / len(token_counts) * 100

print(f"Context window = {CONTEXT_LENGTH} tokens")
print(f"  Fits in 1 window:  {fits_context:,} docs ({pct_fits:.1f}%)")
print(f"  Needs chunking:    {needs_chunking:,} docs ({100-pct_fits:.1f}%)")

# Extrapolate to full dataset (~8M docs)
FULL_DATASET_SIZE = 8_013_769
mean_tokens = token_counts.mean()
total_tokens_est = mean_tokens * FULL_DATASET_SIZE
total_chunks_est = sum(
    max(1, int(np.ceil(tc / CONTEXT_LENGTH))) for tc in token_counts
) / len(token_counts) * FULL_DATASET_SIZE

print(f"\n── Extrapolation to full dataset ({FULL_DATASET_SIZE:,} docs) ──")
print(f"  Estimated total tokens:  {total_tokens_est:,.0f} (~{total_tokens_est/1e9:.1f}B)")
print(f"  Estimated training chunks (@ {CONTEXT_LENGTH} tokens): {total_chunks_est:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# Token count histogram
ax = axes[0]
clip = np.percentile(token_counts, 99)
sns.histplot(token_counts[token_counts <= clip], bins=80, color="mediumpurple",
             edgecolor="white", linewidth=0.3, ax=ax)
ax.axvline(CONTEXT_LENGTH, color="red", ls="--", lw=1.5, label=f"context={CONTEXT_LENGTH}")
ax.axvline(np.median(token_counts), color="black", ls="--", lw=1,
           label=f"median={np.median(token_counts):,.0f}")
ax.set_title("Token count per doc (clipped at P99)")
ax.set_xlabel("Tokens")
ax.legend(fontsize=8)

# Tokens-per-word histogram
ax = axes[1]
clip_tpw = np.percentile(tpw_ratios, 99)
sns.histplot(tpw_ratios[tpw_ratios <= clip_tpw], bins=80, color="goldenrod",
             edgecolor="white", linewidth=0.3, ax=ax)
ax.axvline(np.median(tpw_ratios), color="black", ls="--", lw=1,
           label=f"median={np.median(tpw_ratios):.2f}")
ax.set_title("Tokens per word (high = code/URLs/non-English)")
ax.set_xlabel("Tokens / Word")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Chunks per document: how many 1024-token windows does each doc produce?
chunks_per_doc = np.ceil(token_counts / CONTEXT_LENGTH).astype(int)
chunks_per_doc = np.maximum(chunks_per_doc, 1)  # at least 1 chunk

print(f"Chunks per document (ceil(tokens / {CONTEXT_LENGTH})):")
print(f"  1 chunk:   {(chunks_per_doc == 1).sum():,} docs ({(chunks_per_doc == 1).mean()*100:.1f}%)")
print(f"  2-5:       {((chunks_per_doc >= 2) & (chunks_per_doc <= 5)).sum():,} docs")
print(f"  6-10:      {((chunks_per_doc >= 6) & (chunks_per_doc <= 10)).sum():,} docs")
print(f"  >10:       {(chunks_per_doc > 10).sum():,} docs")
print(f"  Max:       {chunks_per_doc.max()} chunks (doc with {token_counts.max():,} tokens)")

fig, ax = plt.subplots(figsize=(8, 3.5))
clip = int(np.percentile(chunks_per_doc, 99))
sns.histplot(chunks_per_doc[chunks_per_doc <= clip], bins=range(1, clip + 2),
             color="mediumpurple", edgecolor="white", linewidth=0.3, ax=ax, discrete=True)
ax.set_title(f"Training chunks per document (clipped at P99={clip})")
ax.set_xlabel(f"Number of {CONTEXT_LENGTH}-token chunks")
ax.set_ylabel("Documents")
ax.axvline(np.median(chunks_per_doc), color="black", ls="--", lw=1,
           label=f"median={np.median(chunks_per_doc):.0f}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 4 · Text Quality Signals

In [ ]:
URL_RE = re.compile(r'https?://\S+')
HTML_TAG_RE = re.compile(r'<[^>]+>')
BOILERPLATE_PHRASES = [
    "cookie", "privacy policy", "terms of service", "subscribe",
    "sign up", "log in", "all rights reserved", "copyright",
    "click here", "read more", "advertisement",
]
ENGLISH_STOPWORDS = set("the be to of and a in that have i it for not on with he as you do at this but his by from they we say her she or an will my one all would there their what so up out if about who get which go me when make can like time no just him know take people into year your good some could them see other than then now look only come its over think also back after use two how our work first well way even new want because any these give day most us".split())

url_densities = []
html_tag_counts = []
boilerplate_scores = []
non_ascii_ratios = []
stopword_overlaps = []

for t in texts:
    t_lower = t.lower()
    n_chars = max(len(t), 1)
    
    # URL density
    urls = URL_RE.findall(t)
    url_char_count = sum(len(u) for u in urls)
    url_densities.append(url_char_count / n_chars)
    
    # HTML tags
    html_tag_counts.append(len(HTML_TAG_RE.findall(t)))
    
    # Boilerplate phrase matches
    bp_hits = sum(1 for phrase in BOILERPLATE_PHRASES if phrase in t_lower)
    boilerplate_scores.append(bp_hits)
    
    # Non-ASCII ratio
    non_ascii = sum(1 for c in t if ord(c) > 127)
    non_ascii_ratios.append(non_ascii / n_chars)
    
    # English stop-word overlap
    words = set(t_lower.split()[:200])  # first 200 words for speed
    if words:
        stopword_overlaps.append(len(words & ENGLISH_STOPWORDS) / len(words))
    else:
        stopword_overlaps.append(0.0)

url_densities = np.array(url_densities)
html_tag_counts = np.array(html_tag_counts)
boilerplate_scores = np.array(boilerplate_scores)
non_ascii_ratios = np.array(non_ascii_ratios)
stopword_overlaps = np.array(stopword_overlaps)

print("Quality signals computed for all 20K docs.")

In [ ]:
N = len(texts)
quality_flags = {
    "URL density > 10%":        (url_densities > 0.10).sum(),
    "Has HTML tags":            (html_tag_counts > 0).sum(),
    "HTML tags > 10":           (html_tag_counts > 10).sum(),
    "Boilerplate phrases >= 3": (boilerplate_scores >= 3).sum(),
    "Non-ASCII ratio > 5%":     (non_ascii_ratios > 0.05).sum(),
    "Low stopword overlap (<10%)": (stopword_overlaps < 0.10).sum(),
    "Very short docs (<50 chars)":  (char_counts < 50).sum(),
    "Very short docs (<200 chars)": (char_counts < 200).sum(),
}

df_quality = pd.DataFrame([
    {"Flag": k, "Count": v, "% of sample": f"{v/N*100:.2f}%"}
    for k, v in quality_flags.items()
])
df_quality

In [ ]:
def show_flagged(mask, label, n=2):
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        print(f"\n{label}: no documents flagged.")
        return
    print(f"\n{'='*60}")
    print(f"{label} — showing {min(n, len(idxs))} of {len(idxs)} flagged docs")
    print(f"{'='*60}")
    for idx in idxs[:n]:
        preview = texts[idx][:300].replace("\n", "\\n")
        print(f"  [doc {idx}] {preview}...\n")

show_flagged(html_tag_counts > 10, "Heavy HTML")
show_flagged(url_densities > 0.10, "High URL density")
show_flagged(boilerplate_scores >= 3, "Boilerplate-heavy")
show_flagged(stopword_overlaps < 0.10, "Low English stopwords (non-English / code?)")

In [ ]:
# ── Language detection (lightweight heuristic) ──────────────
# Use a broader English word set than stopwords alone. We build a set from
# the 1000 most common English words (covers ~85% of typical English text).
# Docs where <20% of their first 200 words are common English words are
# likely non-English, list/tabular data, or code.

COMMON_ENGLISH = set("""
the be to of and a in that have i it for not on with he as you do at this but
his by from they we say her she or an will my one all would there their what so
up out if about who get which go me when make can like time no just him know
take people into year your good some could them see other than then now look
only come its over think also back after use two how our work first well way
even new want because any these give day most us find here thing many right
very she all her long make thing see him two has look more day could go come
did number sound no most people my over know water than call first who may down
side been now under each just those same much before through help line right too
mean old any same tell boy did three want air play small end put home read hand
port large spell add even land here must big high such follow act why ask men
change went light kind off need house picture try again animal point mother world
near build self earth father head stand own page should country found answer
school grow study still learn plant cover food sun four between state keep eye
never last let thought city tree cross farm hard start might story saw far sea
draw left late run while press close night real life few north open seem together
next white children begin got walk example ease paper group always music will
was been have had are were would could should about their which there other into
more very when only some then also like just over back after much before where
most know through way between both each she all his her from they them been has
its about would make like just know take people year could also back well new
just their were like even because who first after way many then some what only
other into year these than look more day over such make see use find thing long
through each just those same much down before most great where help old any
""".lower().split())

low_english_mask = stopword_overlaps < 0.10
n_flagged = low_english_mask.sum()

# Deeper check: what fraction of words are common English for flagged docs?
english_word_fracs = []
for idx in np.where(low_english_mask)[0]:
    words = texts[idx].lower().split()[:200]
    if words:
        frac = sum(1 for w in words if w.strip(".,!?;:()\"'") in COMMON_ENGLISH) / len(words)
        english_word_fracs.append(frac)
    else:
        english_word_fracs.append(0.0)
english_word_fracs = np.array(english_word_fracs)

print(f"Language check — docs flagged with low stopword overlap (<10%): {n_flagged}")
print(f"  Of these, common-English-word fraction:")
print(f"    <10% (likely non-English/code): {(english_word_fracs < 0.10).sum()}")
print(f"    10-30% (mixed/uncertain):       {((english_word_fracs >= 0.10) & (english_word_fracs < 0.30)).sum()}")
print(f"    >30% (likely English, just unusual): {(english_word_fracs >= 0.30).sum()}")
print(f"\n  → Estimated non-English docs: ~{(english_word_fracs < 0.10).sum()} / {len(texts):,} ({(english_word_fracs < 0.10).sum()/len(texts)*100:.2f}%)")
print(f"  Recommendation: {'Add language filter' if (english_word_fracs < 0.10).sum() > 0 else 'No language filter needed'}")

---
## 5 · Content Pattern Analysis

In [ ]:
# Most common document openings (first 50 chars) — identifies boilerplate headers
openings = Counter(t[:50].strip() for t in texts)
print("Top 10 document openings (first 50 chars):")
print("-" * 60)
for opening, count in openings.most_common(10):
    pct = count / len(texts) * 100
    display = opening.replace("\n", "\\n")
    print(f"  {count:>4}x ({pct:.2f}%) | {display}")

In [ ]:
# Special character / encoding artifact census
ARTIFACTS = {
    "U+FFFD (replacement char)": "\ufffd",
    "BOM (U+FEFF)": "\ufeff",
    "Zero-width space (U+200B)": "\u200b",
    "Zero-width joiner (U+200D)": "\u200d",
    "Zero-width non-joiner (U+200C)": "\u200c",
    "Null byte (U+0000)": "\x00",
    "Soft hyphen (U+00AD)": "\u00ad",
    "Non-breaking space (U+00A0)": "\u00a0",
    "Left-to-right mark (U+200E)": "\u200e",
    "Right-to-left mark (U+200F)": "\u200f",
}

print("Encoding artifact census across 20K documents:")
print("-" * 55)
for name, char in ARTIFACTS.items():
    doc_count = sum(1 for t in texts if char in t)
    total_count = sum(t.count(char) for t in texts)
    pct = doc_count / len(texts) * 100
    print(f"  {name:40s} | {doc_count:>5} docs ({pct:5.2f}%) | {total_count:>7} occurrences")

In [ ]:
# Whitespace & formatting patterns — informs preprocessing step 1
EXCESS_NEWLINES_RE = re.compile(r'\n{3,}')  # 3+ consecutive newlines
TAB_RE = re.compile(r'\t')

has_leading_trailing = sum(1 for t in texts if t != t.strip())
has_excess_newlines = sum(1 for t in texts if EXCESS_NEWLINES_RE.search(t))
has_tabs = sum(1 for t in texts if TAB_RE.search(t))
excess_newline_counts = [len(EXCESS_NEWLINES_RE.findall(t)) for t in texts]
tab_counts = [len(TAB_RE.findall(t)) for t in texts]

print("Whitespace / formatting patterns across 20K documents:")
print("-" * 55)
print(f"  Leading/trailing whitespace:  {has_leading_trailing:>5} docs ({has_leading_trailing/len(texts)*100:.2f}%)")
print(f"  Excessive newlines (3+ in a row): {has_excess_newlines:>5} docs ({has_excess_newlines/len(texts)*100:.2f}%)")
print(f"    → total occurrences: {sum(excess_newline_counts):,}")
print(f"  Contains tab characters:      {has_tabs:>5} docs ({has_tabs/len(texts)*100:.2f}%)")
print(f"    → total tabs: {sum(tab_counts):,}")

---
## 6 · Duplicate Check (Sanity)

In [ ]:
# Quick exact-duplicate check via MD5 (OpenWebText is pre-deduplicated)
hashes = {hashlib.md5(t.encode()).hexdigest() for t in texts}
n_dupes = len(texts) - len(hashes)
print(f"Exact duplicates: {n_dupes:,} / {len(texts):,} docs ({n_dupes/len(texts)*100:.2f}%)")